In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Load Model & Test Data (Ensure paths match your project structure)
rf_model = joblib.load('../models/random_forest_ensemble.pkl')
X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

# Reconstruct feature names from your preprocessor (adjust names based on your encoder)
# Example feature names matching the preprocessing step in Task 1 & 2:
numerical_cols = ['purchase_value', 'age', 'hour_of_day', 'day_of_week', 'time_since_signup', 'device_velocity', 'ip_velocity']
# Dummy feature names for categorical columns after one-hot encoding
feature_names = numerical_cols + [f"cat_{i}" for i in range(X_test.shape[1] - len(numerical_cols))]

# 1. Extract Built-in Feature Importance
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Plot Top 10 Features
plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices[:10]], y=[feature_names[i] for i in indices[:10]], palette='viridis')
plt.title('Top 10 Feature Importances (Built-in MDI)')
plt.xlabel('Mean Decrease in Impurity')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
import shap

# Initialize JavaScript visualization library for SHAP
shap.initjs()

# To speed up SHAP calculation, use a subset of the test set (e.g., 500 samples)
# containing a balanced mix of legitimate and fraud instances
np.random.seed(42)
fraud_indices = np.where(y_test == 1)[0]
legit_indices = np.where(y_test == 0)[0]

sample_indices = np.concatenate([
    np.random.choice(fraud_indices, size=min(100, len(fraud_indices)), replace=False),
    np.random.choice(legit_indices, size=400, replace=False)
])

X_sample = X_test[sample_indices]
y_sample = y_test[sample_indices]

# Compute SHAP Values
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_sample)

# Note: For binary classification in older SHAP versions, shap_values might be a list of two arrays (class 0, class 1).
# We take the values for Class 1 (Fraud).
if isinstance(shap_values, list):
    shap_val_class1 = shap_values[1]
else:
    shap_val_class1 = shap_values

# 1. SHAP Summary Plot (Global Importance)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_val_class1, X_sample, feature_names=feature_names, show=False)
plt.title('SHAP Global Feature Importance & Value Impact', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Obtain predictions from the model
preds = rf_model.predict(X_test)

# Identify target instances
true_positives = np.where((preds == 1) & (y_test == 1))[0]
false_positives = np.where((preds == 1) & (y_test == 0))[0]
false_negatives = np.where((preds == 0) & (y_test == 1))[0]

print(f"Indices available: TP={true_positives[:3]}, FP={false_positives[:3]}, FN={false_negatives[:3]}")

# Pick one of each category for force plots
tp_idx = true_positives[0]
fp_idx = false_positives[0]
fn_idx = false_negatives[0]

# Calculate expected base value
expected_value = explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value

# Helper function to generate and save SHAP force plots
def generate_force_plot(idx, label):
    # Find position of test index in our sample slice, or recalculate SHAP for this specific sample
    single_shap = explainer.shap_values(X_test[idx:idx+1])
    single_shap_class1 = single_shap[1] if isinstance(single_shap, list) else single_shap
    
    plot = shap.force_plot(
        expected_value, 
        single_shap_class1[0], 
        X_test[idx], 
        feature_names=feature_names, 
        matplotlib=True, 
        show=False
    )
    plt.title(f'SHAP Force Plot: {label} (Test Instance {idx})', fontsize=12, pad=30)
    plt.tight_layout()
    plt.show()

# 2. Force Plot: True Positive (Correctly Identified Fraud)
generate_force_plot(tp_idx, "True Positive")

# 3. Force Plot: False Positive (Legitimate Flagged as Fraud)
generate_force_plot(fp_idx, "False Positive")

# 4. Force Plot: False Negative (Missed Fraud)
generate_force_plot(fn_idx, "False Negative")